# 03. Chaining Content Understanding Extraction into an Azure AI Foundry Agent

This notebook combines the two previous notebooks' ideas: it extracts structured fields from an invoice PDF with Azure AI Content Understanding's `prebuilt-invoice` analyzer (as in notebook 01), then hands those extracted fields to a **persisted Azure AI Foundry Agent** (`cloudxeus-support`) via the Responses API's `agent_reference` mechanism, asking the agent to summarize the invoice, judge approval status, flag issues, and recommend next steps. It belongs to Section 07 of this course, the document/content understanding agent section, and is the section's capstone example of chaining extraction into agentic reasoning.

**Difficulty: Advanced**

This notebook is entirely AI-102-relevant: it combines two exam-scoped services — Content Understanding for extraction, and Azure AI Foundry's Agent Service for reasoning over the extracted data — using Entra ID authentication (`DefaultAzureCredential`), the same auth pattern used throughout this repo's `11_azure_ai_foundry/` labs.

## Prerequisites

**pip3 packages:**
```bash
pip3 install azure-ai-contentunderstanding azure-ai-projects azure-identity python-dotenv
```
`azure-ai-projects` and `azure-identity` are already in the repo root `requirements.txt`; `azure-ai-contentunderstanding` is not (see notebook 01's prerequisites).

**Azure resources required:**
- The same Content Understanding-enabled resource used in notebooks 01–02.
- An Azure AI Foundry project (the same kind used throughout `11_azure_ai_foundry/`) with a **persisted Agent** already created and named `cloudxeus-support` (e.g., via the Foundry portal, or the `AIProjectClient`/`AgentsClient` pattern shown in `11_azure_ai_foundry/05_hosted_agent/main.py`). This notebook *calls* an existing agent by name — it does not create one.
- Entra ID auth: you must be logged in via `az login` (or another credential source `DefaultAzureCredential` can discover) with permission to access the Foundry project.

**Environment variables** (add to root `.env`):
```
AZURE_CONTENTUNDERSTANDING_ENDPOINT=https://<your-resource>.services.ai.azure.com/
AZURE_CONTENTUNDERSTANDING_KEY=<your-content-understanding-key>
AZURE_AI_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project-name>
AZURE_AI_AGENT_NAME=cloudxeus-support
```
Note `AZURE_AI_PROJECT_ENDPOINT` reuses this repo's existing convention from `11_azure_ai_foundry/` — it already has the shape `.../api/projects/<project-name>`, matching what this script hardcoded as `PROJECT_ENDPOINT`.

**Local file:** this notebook expects `cloudxeus_sample_invoice.pdf` in the **same folder as this notebook** (`07. Section Code/`) — already present.

## What You'll Learn

- How to chain a Content Understanding extraction result into a prompt for a downstream agent
- The `agent_reference` pattern for invoking a **persisted** Foundry Agent through the Responses API (`openai.responses.create(extra_body={"agent_reference": {...}})`), as opposed to the raw Agents/Threads/Runs API used in `11_azure_ai_foundry/05_hosted_agent/`
- Why this notebook uses `DefaultAzureCredential` (Entra ID) here while notebooks 01–02 used an `AzureKeyCredential` (API key) for Content Understanding — two different services, two independently-chosen auth strategies
- A realistic "extract → reason" document-processing pipeline shape

### Step 1 — Imports, configuration, and the Content Understanding client

Same Content Understanding setup as notebook 01: `AzureKeyCredential` + `ContentUnderstandingClient`. This notebook additionally imports `AIProjectClient` and `DefaultAzureCredential` for the Foundry Agent call that comes later.

💡 **Exam tip:** notice this single script legitimately uses **two different authentication mechanisms** for two different Azure AI services — API key for Content Understanding, Entra ID for the Foundry project/agent. AI-102 expects you to know that auth choice is per-service and per-scenario, not a single repo-wide decision.

🔄 **Alternatives:** if your Content Understanding resource also supports Entra ID auth, you could standardize on `DefaultAzureCredential()` everywhere for consistency and to avoid managing a long-lived API key at all.

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

load_dotenv()

cu_endpoint = os.getenv("AZURE_CONTENTUNDERSTANDING_ENDPOINT")
cu_api_key = os.getenv("AZURE_CONTENTUNDERSTANDING_KEY")

client = ContentUnderstandingClient(
    endpoint=cu_endpoint,
    credential=AzureKeyCredential(cu_api_key),
)


### Step 2 — Extract invoice fields

Identical extraction flow to notebook 01, condensed: submit the sample invoice with `analyzer_id="prebuilt-invoice"`, wait on `poller.result()`, then build a single `invoice_details` string by concatenating every extracted `field_name: field_data` pair with a newline. This flattened string is what gets handed to the agent in Step 3 — the agent never sees the raw PDF, only the already-extracted, already-structured text.

💡 **Exam tip:** this "extract fields → serialize to text → feed to an LLM/agent" hand-off is the general shape of a document-processing pipeline on Azure — Document Intelligence/Content Understanding does the reliable, schema-grounded extraction; the LLM/agent layer handles the judgment calls (summarization, approval logic, anomaly flagging) that a pure extraction model isn't designed to make. Keeping these two responsibilities separate (rather than asking an LLM to also do the raw field extraction from an image) is a best practice AI-102 implicitly rewards.

🔄 **Alternatives:** instead of a flat `field_name: field_data` string, you could pass the fields as structured JSON in the prompt (often more reliable for the model to parse precisely) or attach the original PDF directly to a multimodal-capable agent and skip the explicit extraction step — trading Content Understanding's structured guarantees for the model's own (less predictable) document-reading ability.

In [ ]:
file_path = "cloudxeus_sample_invoice.pdf"

with open(file_path, "rb") as file:
    poller = client.begin_analyze_binary(
        analyzer_id="prebuilt-invoice",
        binary_input=file,
        content_type="application/pdf",
    )

result = poller.result()
print("Analysis completed.")

content = result["contents"][0]
fields = content["fields"]

invoice_details = ""
for field_name, field_data in fields.items():
    invoice_details += f"{field_name}: {field_data}\n"


### Step 3 — Call the persisted Foundry agent by reference

`AIProjectClient(endpoint=..., credential=DefaultAzureCredential())` connects to the Foundry project using Entra ID auth — the exact same pattern used across `11_azure_ai_foundry/`. `project.get_openai_client()` then returns a plain OpenAI-SDK-shaped client pointed at that project, and `openai.responses.create(...)` is called with `extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}}` — this tells Foundry's Responses API to run the request through the **already-created, persisted** agent named `cloudxeus-support`, rather than a stateless one-off chat completion.

The prompt itself embeds `invoice_details` (the flattened extraction from Step 2) and asks the agent for four things: a plain-language summary, an approval status, any issues found, and a recommended next step — turning raw extracted fields into a business-usable review.

💡 **Exam tip:** this `agent_reference` + Responses API pattern is a *different* way of invoking a persisted Foundry Agent than the Agents/Threads/Runs API shown in `11_azure_ai_foundry/05_hosted_agent/main.py` (which explicitly creates a thread, adds messages, creates a run, and polls run status). Both ultimately drive the same underlying persisted Agent resource; `agent_reference` is a more compact, Responses-API-native way to invoke it by name for a single-turn call, without manually managing thread/run objects yourself. Know that Foundry offers more than one API surface for calling the same agent.

🔄 **Alternatives:** use the classic Agents/Threads/Runs flow (`AgentsClient`, `create_thread`, `create_message`, `create_run`, poll until complete) instead of `agent_reference`, if you need multi-turn conversational state tied to a thread rather than a single-shot Responses call.

In [ ]:
PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
AGENT_NAME = os.getenv("AZURE_AI_AGENT_NAME", "cloudxeus-support")

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

openai = project.get_openai_client()

response = openai.responses.create(
    extra_body={
        "agent_reference": {
            "name": AGENT_NAME,
            "type": "agent_reference",
        }
    },
    input=f"""
Review this invoice using only the extracted fields below.

Your task:
1. Summarize the invoice in business-friendly language.
2. Provide an approval status.
3. Identify any issues found.
4. Recommend the next step.

Extracted invoice fields:
{invoice_details}
""",
)

print("\n--- Agent Invoice Review ---")
print(response.output_text)


## Summary

This notebook chained two Azure AI services together: Content Understanding's `prebuilt-invoice` analyzer extracted structured fields from a sample invoice PDF, and that extracted text was then handed as context to a persisted Azure AI Foundry Agent (`cloudxeus-support`) via the Responses API's `agent_reference` mechanism. The printed output is the agent's business-friendly review — a summary, approval status, flagged issues, and a recommended next step — synthesized entirely from the structured fields, not the raw PDF. This is a complete, realistic "extract then reason" document-automation pipeline shape.

## Try It Yourself

1. **Easy:** Change the prompt's four numbered instructions to ask for a fifth thing, e.g. "5. Estimate the risk of late payment based on the due date," and see how the agent's output adapts.
2. **Intermediate:** Pass `invoice_details` as structured JSON instead of a flattened `field_name: field_data` string (e.g., `json.dumps(fields, default=str, indent=2)`) and compare whether the agent's response quality changes.
3. **Advanced:** Rebuild this same pipeline against the Agents/Threads/Runs API (`11_azure_ai_foundry/05_hosted_agent/main.py`'s pattern) instead of `agent_reference`, so each invoice review becomes a message in a persistent thread — then process several invoices in sequence and ask the agent a follow-up question that references an earlier invoice, testing whether thread-based memory adds real value for this use case.